# Notebook 01: KDD Cup 99, the data and the protocol

**Class 03_01.** KDD Cup 99 is the oldest network-intrusion benchmark: 41 features per TCP/IP *connection*
summarised from DARPA 1998 traffic, labelled *normal* or one of 22 attacks in four categories:

| category | what it does | examples |
|:--|:--|:--|
| **dos** | denial of service: floods a service | neptune (SYN flood), smurf, back |
| **probe** | scans hosts and ports | satan, ipsweep, portsweep, nmap |
| **r2l** | remote to local: gets a local account from outside | guess_passwd, warezclient, imap |
| **u2r** | user to root: escalates privileges | buffer_overflow, rootkit, perl |

It is also a famous **bad** data set (Tavallaee et al. 2009; McHugh 2000): most rows are duplicates, the attack
share is unrealistically high, and the traffic is from 1998. We use it *because* its flaws teach the protocol:
deduplicate, set a realistic base rate, threshold on normal data, report per category.

In [ ]:
import sys

import numpy as np
import polars as pl
from sklearn.metrics import precision_recall_curve, roc_curve

sys.path[:0] = [".", "../../src"]  # this folder and the generic libraries of the repository
import aasplot as ap
import anomlib as an
from aasplot import BLUE, GREEN, RED

an.table_config()
ap.setup("03")
raw = an.load_kdd(dedupe=False)
df = an.load_kdd()
print(f"rows: {len(raw):,} -> unique {len(df):,} ({1 - len(df) / len(raw):.1%} duplicates)")
cmp = []
for name, d in [("raw", raw), ("deduplicated", df)]:
    vc = d["category"].value_counts()
    cmp.append({"data": name, **{c: int(vc.get(c, 0)) for c in ["normal", *an.CATEGORIES]},
                "attack share": round(1 - vc["normal"] / len(d), 3)})  # fmt: skip
print(pl.DataFrame(cmp))

In [ ]:
CATS = an.CATEGORIES


def dedup(raw: dict, dedup_counts: dict) -> None:
    """raw, dedup_counts: rows per category (normal, dos, probe, r2l, u2r)."""
    f, ax = ap.fig("full", 1.1)
    for y, c in ((1, raw), (0, dedup_counts)):
        a = sum(c[k] for k in CATS)
        ax.barh(y, c["normal"] / 1e3, color=GREEN, height=0.55, label="normal" if y else None)
        ax.barh(y, a / 1e3, left=c["normal"] / 1e3, color=RED, height=0.55, label="attacks" if y else None)
        ax.text((c["normal"] + a) / 1e3 + 6, y, f"{100 * a / (a + c['normal']):.0f}% attacks", va="center", fontsize=7)
    ax.set_yticks([1, 0], ["raw (10% file)", "deduplicated"])
    ax.set(xlabel="connections (thousands)", xlim=(0, 620))
    ax.legend(loc="lower right", ncol=2)
    ax.grid(axis="y", visible=False)
    ap.save(f, "dedup")


dedup(*({c: r[c] for c in ["normal", *an.CATEGORIES]} for r in cmp))

**Lesson 1.** In the raw file 80% of the rows are attacks, mostly copies of two floods (smurf, neptune). A detector
evaluated there is rewarded for recognising two DoS patterns. After deduplication normal traffic is the majority,
but the attack share (40%) is still far from any real network.

## The features

Three groups: **basic** (duration, protocol, service, flag, bytes), **content** (failed logins, root shells, file
creations: read from the payload) and **traffic** (counts of connections to the same host or service in the last
2 seconds or 100 connections). Three columns are categorical.

In [ ]:
print("categorical:", {c: df[c].nunique() for c in an.CATEGORICAL})
print(df.groupby("category")[["duration", "src_bytes", "dst_bytes", "count", "serror_rate", "num_failed_logins"]]
      .median().round(2))  # fmt: skip

Bytes and counts are heavy-tailed (a few connections move megabytes): we take $\log(1+x)$ of them, one-hot encode
the three categorical columns (71 features in total), and z-scale with the mean and standard deviation of the
**normal training rows only**.

## The protocol (novelty detection)

* Normal rows: 60% train, 20% validation, 20% test. **Attacks never appear in training.**
* Test: the normal test rows plus up to 100 attacks per category (u2r has only 52 unique rows: half go to the test,
  half to a *pool* used later as labelled examples). About 1.8% of the test set is attack.
* The threshold is the 99% quantile of the scores on the normal validation set (about 1% false alarms).

In [ ]:
sp = an.kdd_split(df, seed=0)
print({k: len(getattr(sp, k)) for k in ["train", "val", "test", "pool"]}, "features:", sp.X("train").shape[1])
print("test attacks per category:", sp.test["category"].value_counts().to_dict())
print(f"attack share of the test set: {sp.y_test.mean():.3%}")

## ROC or precision-recall?

Two curves summarise a score without choosing a threshold:
* **ROC**: true-positive rate against false-positive rate; AUROC = probability that a random attack scores above a
  random normal row. It ignores the base rate.
* **Precision-recall**: precision against recall; **AUPRC** (average precision) depends on the base rate. A random
  score has AUROC 0.5 but AUPRC equal to the attack share.

Same Mahalanobis detector, same scores, two test sets: the realistic one (1.8% attacks) and a "balanced" one.

In [ ]:
Xtr, Xva, Xte, y = sp.X("train"), sp.X("val"), sp.X("test"), sp.y_test
det = an.Mahalanobis().fit(Xtr)
s_te, s_va = det.score(Xte), det.score(Xva)
Xp = sp.X("pool")
s_bal = np.r_[s_te[y == 0], det.score(Xp[: (y == 0).sum()])]
y_bal = np.r_[np.zeros((y == 0).sum()), np.ones(min(len(Xp), (y == 0).sum()))]
rows = [{"test set": "1.8% attacks", **an.evaluate(s_te, y, s_va)},
        {"test set": "balanced (pool, 95% dos)", **an.evaluate(s_bal, y_bal, s_va)}]  # fmt: skip
print(pl.DataFrame(rows))

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score


def roc_vs_pr(y_test, s_test, y_bal, s_bal) -> None:
    """The same scores on the realistic test set and on a balanced one: ROC barely changes, precision does."""
    f, axs = ap.fig("full", 1.45, ncols=2, gridspec_kw={"wspace": 0.3})
    au = {}
    for lab, col, y, s in (("1.8% attacks", RED, y_test, s_test), ("balanced pool", BLUE, y_bal, s_bal)):
        fpr, tpr, _ = roc_curve(y, s)
        pr, rc, _ = precision_recall_curve(y, s)
        axs[0].plot(fpr, tpr, color=col, label=lab)
        axs[1].plot(rc, pr, color=col, label=lab)
        au[lab] = (roc_auc_score(y, s), average_precision_score(y, s))
    a, b = au["1.8% attacks"], au["balanced pool"]
    axs[0].set(
        xscale="log",
        xlim=(1e-4, 1),
        xlabel="false-alarm rate (log)",
        ylabel="recall",
        title=f"ROC: AUROC {a[0]:.3f} / {b[0]:.3f}",
    )
    axs[1].set(xlabel="recall", ylabel="precision", title=f"PR: AUPRC {a[1]:.3f} / {b[1]:.3f}")
    axs[0].legend(loc="lower right", fontsize=7)
    axs[1].legend(loc="lower left", fontsize=7)
    ap.save(f, "roc_vs_pr")


roc_vs_pr(y, s_te, y_bal, s_bal)

**Lesson 2.** The ROC curve barely changes with the base rate; precision collapses when attacks are rare. Report
AUPRC (and the precision at your threshold) for a detector that will face rare events, and always state the base
rate of the test set.

**Lesson 3 (what to report).** At the 1% validation threshold: recall, precision, F1, the *measured* FPR on the
normal test rows (it should be close to 1%), and the recall **per category**, because one easy category (dos)
can hide a blind spot (r2l).